In [ ]:
import random
from datasets import load_dataset, Dataset, IterableDataset

# ============================================================================
# 📚 데이터셋 소개: NoNameFactory/korean_safe_conversation
# 🇰🇷 한글 제목: 한국어 안전 대화 데이터셋
# 📜 의미/설명: 이 데이터셋은 안전하고 적절한 한국어 대화 패턴을 학습하는 데 사용되는 대화 로그 모음입니다.
# 💡 목표: 초보자가 대화 구조를 분석하고, 역할(Role)별 발화 패턴을 탐색하는 재미있는 실습을 진행해 봅시다!
# ============================================================================

# 상수 설정
DATASET_NAME = "NoNameFactory/korean_safe_conversation"
SAMPLE_COUNT = 5  # 실습에 사용할 샘플 개수 (너무 많으면 느립니다!)

print("🤖 안녕! 파이썬으로 AI 데이터셋 탐험에 오신 걸 환영해요! 😊")
print("오늘은 대화 로그 데이터셋의 구조를 파헤쳐 보고, 실제 대화의 패턴을 분석하는 재미있는 미션을 수행할 거예요.")
print("-" * 60)


# ----------------------------------------------------------------------------
# 📌 1. 데이터 로딩 (스트리밍 vs. 전체 로드)
# ----------------------------------------------------------------------------
dataset = None

# 1. 스트리밍 로드 시도 (메모리 효율적, 권장 방식)
try:
    print("🚀 1단계: 스트리밍 모드(streaming=True)로 데이터셋을 로드해 봅니다. (빠른 탐색 시작!)")
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 성공적으로 스트리밍 데이터셋을 로드했습니다. 대용량 데이터 처리 준비 완료!")

except Exception as e:
    print(f"⚠️ 스트리밍 로드에 실패했습니다. 오류: {e}")
    # 2. 스트리밍 실패 시, 작은 배치만 다운로드하여 로드합니다.
    print("💾 소규모 다운로드 모드(streaming=False)로 전환합니다.")
    try:
        dataset = load_dataset(DATASET_NAME, split='train', streaming=False)
        print("✅ 작은 배치(Dataset) 형태로 데이터를 로드했습니다. 걱정 마세요, 진행할 수 있어요!")
    except Exception as e2:
        print(f"❌ 모든 로드 시도에 실패했습니다. 데이터셋 이름이나 연결을 확인해주세요. 오류: {e2}")

# ----------------------------------------------------------------------------
# 📌 2. 샘플 데이터 추출 및 준비
# ----------------------------------------------------------------------------

# streaming 모드(IterableDataset)와 일반 모드(Dataset)에 맞는 샘플 추출 패턴을 사용해야 합니다.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)
    print(f"\n✨ {SAMPLE_COUNT}개의 샘플 데이터를 스트리밍 방식으로 준비합니다.")
    sampled_dataset_iterator = dataset.take(SAMPLE_COUNT)
    # 메모리에 올리기 위해 리스트로 변환 (초보자에게는 list로 보는 것이 직관적입니다)
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 데이터셋 (Dataset)인 경우
    sample_data_list = [dataset[i] for i in range(min(SAMPLE_COUNT, len(dataset)))]

print(f"✅ 총 {len(sample_data_list)}개의 샘플을 분석 대상으로 선정했습니다. (✨ 상위 {SAMPLE_COUNT}개만 사용)")
print("-" * 60)


# ----------------------------------------------------------------------------
# 🤖 3. 창의적 AI 실습: 대화 턴(Turn) 패턴 분석기
# ----------------------------------------------------------------------------
print("🤖 미션: 대화 로그에서 '총 몇 번의 발화 교환(턴)'이 있었는지, 그리고 '어떤 역할(Role)'이 주로 이야기했는지 분석해 봅시다!")

# 분석 결과를 저장할 리스트
all_turn_counts = []
all_role_counts = []

# 데이터 분석 루프 시작
for i, sample in enumerate(sample_data_list):
    print(f"\n📚 [샘플 #{i+1}] 분석 시작...")
    
    # 데이터셋 구조 접근: 'conversations' 필드 안에 리스트가 들어 있습니다.
    if 'conversations' not in sample:
        print("⚠️ 이 샘플은 대화 구조를 가지고 있지 않아 분석할 수 없습니다.")
        continue
        
    conversations = sample['conversations']
    
    turn_count = 0
    role_frequency = {} # 역할별 발화 횟수를 저장할 딕셔너리
    
    print(f"   -> 총 {len(conversations)}개의 발화 기록을 발견했습니다.")
    
    # 대화 기록을 순회하며 분석 수행
    for conversation_turn in conversations:
        # 1. 발화 횟수 증가 (Turn Count)
        turn_count += 1
        
        # 2. 역할 추출 (Role Extraction)
        role = conversation_turn.get('from', '알 수 없는 역할')
        
        # 3. 역할 빈도 카운트
        role_frequency[role] = role_frequency.get(role, 0) + 1
        
        # (Optional) 내용 출력: print(f"   - {role}: {conversation_turn['value'][:30]}...")
        
    # 결과 저장
    all_turn_counts.append(turn_count)
    all_role_counts.append(role_frequency)
    
    # 최종 요약 출력
    print(f"   ✨ 분석 결과: 이 대화는 총 {turn_count}번의 턴으로 이루어져 있습니다.")
    print("   👑 역할별 발화 패턴:")
    for role, count in role_frequency.items():
        print(f"      - '{role}' 역할: {count}회 발화 (가장 많이 말하는 사람일 수 있어요!)")

print("\n" + "=" * 60)
print("✨ 분석 완료! 종합 통계 요약:")

# ----------------------------------------------------------------------------
# 📊 4. 종합 결과 (Quantitative Summary)
# ----------------------------------------------------------------------------
total_turns_sum = sum(all_turn_counts)
average_turns = total_turns_sum / len(sample_data_list) if sample_data_list else 0

# 가장 많이 사용된 역할을 전체 샘플에서 찾기 (Simple Majority Vote)
global_role_counter = {}
for role_freq in all_role_counts:
    for role, count in role_freq.items():
        global_role_counter[role] = global_role_counter.get(role, 0) + count

print(f"📊 전체 {len(sample_data_list)}개 샘플에서 기록된 총 발화 턴(Total Turns): {total_turns_sum}번")
print(f"📊 평균 대화 턴(Average Turn per sample): {average_turns:.2f} 턴")
print("\n💡 전체 샘플에서 가장 많이 관찰된 발화 역할을 상위 3개까지 뽑아볼게요!")

# 역할 빈도수를 기준으로 정렬 및 상위 3개 출력
sorted_roles = sorted(global_role_counter.items(), key=lambda item: item[1], reverse=True)

for i, (role, count) in enumerate(sorted_roles):
    if i >= 3:
        break
    print(f"   🥇 {i+1}위 역할: '{role}' (총 {count}회 발화)")

print("\n✅ 축하합니다! 🎉")
print("데이터셋의 대화 패턴 분석을 성공적으로 완료했습니다. 이제 이 패턴 분석을 통해...")
print("🤖 다음 단계로, 실제 'AI가 어떤 역할로 말해야 할지' 예측하는 LLM 프롬프트의 기반 지식으로 활용할 수 있답니다!")